# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashalaf/flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Auditing `outputs/model_report.md`, the reference pipeline's own results report on this dataset, constructive tone throughout, the goal is the next level of rigor, not a scalp.

**Finding A: "random_forest reaches Precision@50 = 0.740, a ~3x lift over the baseline_rules (0.240), validated on a client_holdout split."**

- *Where does the label come from?* `is_declining_label`, from `trend_direction`, same as this notebook's own model, an observed outcome, not a hand-defined rule.
- *Does the validation design carry the claim?* The report labels its split "client_holdout," but doesn't show the client-overlap count that would prove it. That check matters a lot here: rerunning this same task honestly below (Section 2) shows a genuinely zero-overlap grouped split scoring 0.600, while a naive random split (31 clients leaking across train/test) scores 0.760, suspiciously close to the report's claimed 0.740. That's not proof the report's split was leaky, only the report's own code would settle that, but it's a concrete, checkable reason to ask for the overlap count before trusting the ~3x-lift headline as-is.

**Finding B: top feature importances include `avg_position` (10.90%) and reason codes like `low_ctr_visible_page` built on `ctr`.**

- *Where does the label come from?* Same `is_declining_label`.
- *Does the validation design carry the claim?* ML-05's leakage audit (this repo, same dataset) found that `avg_position` and `ctr` are 90-day aggregates with no safe prev-30-day-only version, and that `_last_30d` sub-windows of the same metrics consistently correlated harder with the label than `_prev_30d` ones, a real, measured window-overlap risk, not a hypothetical one. That's exactly why this notebook's own feature set excludes both columns entirely. The reference model uses both directly as top features, its precision numbers may be partly inflated by the same window-overlap effect this repo's own ML-05 work found and removed.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashalaf/flyrank-internship"
REPO_DIR = "flyrank-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd
import numpy as np

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), \
    "starter CSV not found -- are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Reference report claims, for context (from outputs/model_report.md):")
print("  random_forest precision@50: 0.740")
print("  baseline_rules precision@50: 0.240")
print("  split strategy stated: client_holdout (overlap count not shown in the report)")
print("  top features include: avg_position (10.90%), ctr (3.30%)")

Reference report claims, for context (from outputs/model_report.md):
  random_forest precision@50: 0.740
  baseline_rules precision@50: 0.240
  split strategy stated: client_holdout (overlap count not shown in the report)
  top features include: avg_position (10.90%), ctr (3.30%)


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Re-running this notebook's own Week-5 (ML-08) Logistic Regression under two splits on the exact same leakage-safe feature set, before (naive random) and after (grouped by `client_id`), with the overlap count shown for both, not just claimed.

In [2]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

static_numeric = ["word_count", "char_count", "search_volume", "competition", "cpc",
                   "content_age_days", "days_since_last_update"]
static_categorical = ["content_type", "main_intent", "competition_level", "age_tier", "freshness_tier"]
safe_traffic = ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
numeric_cols = static_numeric + safe_traffic
X_numeric = df[numeric_cols].fillna(-1)
X_categorical = pd.get_dummies(df[static_categorical].fillna("unknown"), prefix=static_categorical)
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df["is_declining_label"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K = 50

# BEFORE: naive random split
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler_r = StandardScaler()
lr_r = LogisticRegression(max_iter=2000, random_state=42)
lr_r.fit(scaler_r.fit_transform(Xtr_r), ytr_r)
p50_random = precision_at_k(lr_r.predict_proba(scaler_r.transform(Xte_r))[:, 1], yte_r.values, K)
overlap_random = len(set(df.iloc[Xtr_r.index]["client_id"]) & set(df.iloc[Xte_r.index]["client_id"]))

# AFTER: grouped by client_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
Xtr_g, Xte_g = X.iloc[train_idx], X.iloc[test_idx]
ytr_g, yte_g = y.iloc[train_idx], y.iloc[test_idx]
scaler_g = StandardScaler()
lr_g = LogisticRegression(max_iter=2000, random_state=42)
lr_g.fit(scaler_g.fit_transform(Xtr_g), ytr_g)
p50_grouped = precision_at_k(lr_g.predict_proba(scaler_g.transform(Xte_g))[:, 1], yte_g.values, K)
overlap_grouped = len(set(df.iloc[train_idx]["client_id"]) & set(df.iloc[test_idx]["client_id"]))

print("BEFORE (naive random split):")
print(f"  precision@{K}: {p50_random:.3f}, client overlap between train/test: {overlap_random}")
print("\nAFTER (grouped by client_id):")
print(f"  precision@{K}: {p50_grouped:.3f}, client overlap between train/test: {overlap_grouped}")
print(f"\ndrop from an honest split: {p50_random - p50_grouped:.3f}")
print(f"note: the 'before' number ({p50_random:.3f}) sits suspiciously close to the reference")
print(f"report's claimed 0.740, exactly the kind of gap Finding A raised a question about above.")

BEFORE (naive random split):
  precision@50: 0.760, client overlap between train/test: 31

AFTER (grouped by client_id):
  precision@50: 0.600, client overlap between train/test: 0

drop from an honest split: 0.160
note: the 'before' number (0.760) sits suspiciously close to the reference
report's claimed 0.740, exactly the kind of gap Finding A raised a question about above.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Same confession test as ML-05, rerun on this notebook's final feature set to confirm nothing regressed since then.

In [3]:
X_leaky = X.copy()
X_leaky["trend_pct_LEAKY"] = (df["trend_pct"].fillna(0) - df["trend_pct"].fillna(0).mean()) / (df["trend_pct"].fillna(0).std() + 1e-9)

from sklearn.metrics import roc_auc_score

Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(X_leaky, y, test_size=0.2, random_state=42, stratify=y)
scaler_l = StandardScaler()
leaky_model = LogisticRegression(max_iter=2000, random_state=42)
leaky_model.fit(scaler_l.fit_transform(Xtr_l), ytr_l)
auc_leaky = roc_auc_score(yte_l, leaky_model.predict_proba(scaler_l.transform(Xte_l))[:, 1])

scaler_h = StandardScaler()
honest_model = LogisticRegression(max_iter=2000, random_state=42)
honest_model.fit(scaler_h.fit_transform(Xtr_r), ytr_r)
auc_honest = roc_auc_score(yte_r, honest_model.predict_proba(scaler_h.transform(Xte_r))[:, 1])

print(f"honest AUC (final feature set, no trend_pct): {auc_honest:.3f}")
print(f"AUC with trend_pct deliberately re-added: {auc_leaky:.3f}")
print(f"-> still confesses the same way it did in ML-05, final feature set has not regressed.")

# confirm the excluded columns are still excluded
excluded = {"avg_position", "ctr", "trend_direction", "trend_pct", "impressions_90d",
            "impressions_last_30d", "clicks_last_30d", "sessions_last_30d"}
print(f"\nexcluded columns still absent from X: {excluded.isdisjoint(set(X.columns))}")

honest AUC (final feature set, no trend_pct): 0.642
AUC with trend_pct deliberately re-added: 1.000
-> still confesses the same way it did in ML-05, final feature set has not regressed.

excluded columns still absent from X: True


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Boldest sentence, from ML-08 (Week 5):** *"Logistic Regression beat both baselines and the base rate."*

Rewritten on the claim ladder: this is a validated model outperforming a comparison, evaluated out-of-sample under a grouped split, which is the "the model ranks/flags... at precision@K of..." tier, not causal, not universal.

**Rewrite:** *"Under a client-grouped, held-out split, the logistic regression model ranked content at 0.600 precision@50 on this dataset, higher than both the recomputed rule baseline (0.320) and the base rate (0.511) on the same test rows. This is a decision-support signal for this dataset and split, not a guarantee it generalizes to a different client mix or time period, and it should be re-checked if the underlying traffic patterns shift."*

**What changed and why:** dropped "beat" (competitive/causal-sounding) for "ranked... higher than" (a measured comparison); named the exact split and metric instead of leaving them implicit; added the decision-support scope limit explicitly, per the skill, cross-sectional, single-dataset results never support "this will work everywhere," only "this looked worth using here, on this evidence."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.